# Amazon ML Challenge 2026: Business Entity Resolution
## Phase 1: Exploratory Data Analysis (EDA) & Data Understanding

**Objective:**
Inspect and understand the multi-source business entity resolution dataset across Source 1 (deduplicated reference entities), Source 2 (noisy entities), and Source 3 (noisy entities), as well as the Ground Truth annotations and Test sets.

**Scope of Phase 1:**
- Analyze shapes, schemas, and data types
- Analyze missing values and empty string anomalies
- Inspect country distributions and open-set requirements (including France in the test set)
- Examine text length statistics (names and addresses)
- Uncover empirical noise patterns from matching pairs
- Benchmark Phase 1 preprocessing functions (`src/preprocessing.py`)
- Document actionable findings for **Phase 2 (Blocking)** and **Phase 3 (Feature Engineering)**


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root and src are accessible
current_dir = os.getcwd()
for candidate in [current_dir, os.path.dirname(current_dir), os.path.join(current_dir, "student_resource"), os.path.join(os.path.dirname(current_dir), "student_resource")]:
    if os.path.exists(os.path.join(candidate, "src", "preprocessing.py")):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)
        break

from src.preprocessing import (
    normalize_text,
    normalize_business_name,
    normalize_address,
    normalize_country,
    extract_name_tokens,
    extract_address_tokens,
    preprocess_dataframe
)

# Plotting configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("Environment initialized successfully.")


## 1. Dataset Overview & File Structure

The challenge provides three sources of business data plus ground truth for training:
- **`train_source1.tsv`**: Reference deduplicated business entities (primary query entities).
- **`train_source2.tsv`**: Noisy web-scale business records.
- **`train_source3.tsv`**: Noisy web-scale business records.
- **`train_ground_truth.tsv`**: Ground truth matches linking each Source 1 entity to matching Source 2 and Source 3 records.


In [ ]:
# Locate datasets flexibly
def find_dataset_dir():
    candidates = [
        "dataset",
        "student_resource/dataset",
        "../dataset",
        "../student_resource/dataset"
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train", "train_source1.tsv")):
            return c
    raise FileNotFoundError("Could not locate dataset directory.")

dataset_dir = find_dataset_dir()
train_dir = os.path.join(dataset_dir, "train")
test_dir = os.path.join(dataset_dir, "test")

print(f"Dataset root directory: {os.path.abspath(dataset_dir)}")


### 1.1 Dataset Scale (Total Row Counts)
Entity Resolution at this scale requires memory-conscious processing.


In [ ]:
def get_line_count(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return sum(1 for _ in f) - 1  # subtract header

file_counts = {
    "Train Source 1": get_line_count(os.path.join(train_dir, "train_source1.tsv")),
    "Train Source 2": get_line_count(os.path.join(train_dir, "train_source2.tsv")),
    "Train Source 3": get_line_count(os.path.join(train_dir, "train_source3.tsv")),
    "Train Ground Truth": get_line_count(os.path.join(train_dir, "train_ground_truth.tsv")),
    "Test Source 1": get_line_count(os.path.join(test_dir, "test_source1.tsv")),
    "Test Source 2": get_line_count(os.path.join(test_dir, "test_source2.tsv")),
    "Test Source 3": get_line_count(os.path.join(test_dir, "test_source3.tsv")),
}

counts_df = pd.DataFrame(list(file_counts.items()), columns=["Dataset Split", "Total Records"])
counts_df["Total Records (Formatted)"] = counts_df["Total Records"].apply(lambda x: f"{x:,}")
display(counts_df)


## 2. Schema Inspection & Data Types

Let's load representative samples (100,000 records each) from each training source to examine schemas, types, and distributions.


In [ ]:
SAMPLE_SIZE = 100_000

df_s1 = pd.read_csv(os.path.join(train_dir, "train_source1.tsv"), sep="\t", nrows=SAMPLE_SIZE)
df_s2 = pd.read_csv(os.path.join(train_dir, "train_source2.tsv"), sep="\t", nrows=SAMPLE_SIZE)
df_s3 = pd.read_csv(os.path.join(train_dir, "train_source3.tsv"), sep="\t", nrows=SAMPLE_SIZE)

print(f"Sample loaded: {len(df_s1):,} rows from each source.")


In [ ]:
# Schema overview
schema_info = []
for name, df in [("Source 1", df_s1), ("Source 2", df_s2), ("Source 3", df_s3)]:
    for col in df.columns:
        schema_info.append({
            "Source": name,
            "Column": col,
            "Dtype": str(df[col].dtype),
            "Non-Null Count": df[col].notnull().sum(),
            "Sample Value": str(df[col].dropna().iloc[0])[:40]
        })

schema_df = pd.DataFrame(schema_info)
display(schema_df)


## 3. Missing Value & Anomaly Analysis

Understanding where data is absent is vital. A critical discovery from our analysis is that **`business_address` contains missing values (NaN) in Source 2 and Source 3 (~3.3%)**, whereas Source 1 has zero missing addresses.


In [ ]:
missing_summary = []
for name, df in [("Source 1", df_s1), ("Source 2", df_s2), ("Source 3", df_s3)]:
    for col in df.columns:
        null_count = df[col].isnull().sum()
        empty_str_count = (df[col].dropna().astype(str).str.strip() == "").sum()
        missing_summary.append({
            "Source": name,
            "Column": col,
            "Null Count": null_count,
            "Null Rate (%)": round((null_count / len(df)) * 100, 2),
            "Empty String Count": empty_str_count
        })

missing_df = pd.DataFrame(missing_summary)
display(missing_df)


## 4. Country Distribution & Open-Set Analysis

**Key Requirement:**
The competition specifies:
- Training data contains `US` and `India`.
- Test data introduces **`France`** as a third country.
- The pipeline must treat country as an **open set** of string labels, never hard-coding `{US, India}`.


In [ ]:
# Train country distributions
country_data = []
for name, df in [("Train S1", df_s1), ("Train S2", df_s2), ("Train S3", df_s3)]:
    vc = df["country"].value_counts(normalize=True) * 100
    for country, pct in vc.items():
        country_data.append({"Source": name, "Country": country, "Percentage": pct})

# Inspect test country distributions (sample 50,000)
for t_file, label in [("test_source1.tsv", "Test S1"), ("test_source2.tsv", "Test S2"), ("test_source3.tsv", "Test S3")]:
    t_df = pd.read_csv(os.path.join(test_dir, t_file), sep="\t", nrows=50_000)
    vc = t_df["country"].value_counts(normalize=True) * 100
    for country, pct in vc.items():
        country_data.append({"Source": label, "Country": country, "Percentage": pct})

country_df = pd.DataFrame(country_data)

plt.figure(figsize=(11, 5))
sns.barplot(data=country_df, x="Source", y="Percentage", hue="Country")
plt.title("Country Distribution: Training vs. Test Sets")
plt.ylabel("Percentage of Records (%)")
plt.xlabel("Dataset Source")
plt.legend(title="Country")
plt.tight_layout()
plt.show()


## 5. Text Length Analysis (Names & Addresses)

Analyzing character and word count distributions provides guidance for:
- N-gram sizes for candidate generation / blocking
- Minimum and maximum sequence lengths
- Handling unusually long addresses (e.g. multi-line premises, landmark descriptions)


In [ ]:
# Compute name lengths
df_s1["name_len"] = df_s1["business_name"].dropna().str.len()
df_s2["name_len"] = df_s2["business_name"].dropna().str.len()
df_s3["name_len"] = df_s3["business_name"].dropna().str.len()

# Compute address lengths
df_s1["addr_len"] = df_s1["business_address"].dropna().str.len()
df_s2["addr_len"] = df_s2["business_address"].dropna().str.len()
df_s3["addr_len"] = df_s3["business_address"].dropna().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Business name length distribution
sns.kdeplot(df_s1["name_len"], ax=axes[0], label="Source 1 (Ref)", fill=True, alpha=0.3)
sns.kdeplot(df_s2["name_len"], ax=axes[0], label="Source 2", fill=True, alpha=0.3)
sns.kdeplot(df_s3["name_len"], ax=axes[0], label="Source 3", fill=True, alpha=0.3)
axes[0].set_title("Business Name Length Distribution (Characters)")
axes[0].set_xlabel("Character Count")
axes[0].set_xlim(0, 100)
axes[0].legend()

# Address length distribution
sns.kdeplot(df_s1["addr_len"], ax=axes[1], label="Source 1 (Ref)", fill=True, alpha=0.3)
sns.kdeplot(df_s2["addr_len"], ax=axes[1], label="Source 2", fill=True, alpha=0.3)
sns.kdeplot(df_s3["addr_len"], ax=axes[1], label="Source 3", fill=True, alpha=0.3)
axes[1].set_title("Business Address Length Distribution (Characters)")
axes[1].set_xlabel("Character Count")
axes[1].set_xlim(0, 180)
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Ground Truth Analysis & Singleton Distribution

A Source 1 entity may match zero, one, or multiple entities from Source 2 and Source 3.
Because the evaluation metric is **Macro-averaged F_0.5**, correctly predicting singletons (entities with zero matches) yields a score of **1.0** on that entity.


In [ ]:
gt_sample = pd.read_csv(os.path.join(train_dir, "train_ground_truth.tsv"), sep="\t", nrows=100_000)

def count_matches(val):
    if pd.isna(val) or not str(val).strip():
        return 0
    return len(str(val).split(","))

gt_sample["match_count"] = gt_sample["matched_entity_ids"].apply(count_matches)

print("Match multiplicity distribution in 100k sample:")
print(gt_sample["match_count"].value_counts(normalize=True).sort_index() * 100)

plt.figure(figsize=(9, 4))
match_counts = gt_sample["match_count"].value_counts().sort_index()
match_counts[:8].plot(kind="bar", color="teal")
plt.title("Distribution of Matches per Source 1 Entity")
plt.xlabel("Number of Matched Entities")
plt.ylabel("Source 1 Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Real Noise Patterns from Ground Truth Pairs

Let's examine true matching entity pairs directly from ground truth to categorize observed real-world noise.


In [ ]:
# Load an index of matching pairs
s1_subset = pd.read_csv(os.path.join(train_dir, "train_source1.tsv"), sep="\t", nrows=10_000)
gt_subset = pd.read_csv(os.path.join(train_dir, "train_ground_truth.tsv"), sep="\t", nrows=10_000)
s2_subset = pd.read_csv(os.path.join(train_dir, "train_source2.tsv"), sep="\t", nrows=25_000)
s3_subset = pd.read_csv(os.path.join(train_dir, "train_source3.tsv"), sep="\t", nrows=25_000)

s2_dict = s2_subset.set_index("entity_id").to_dict("index")
s3_dict = s3_subset.set_index("entity_id").to_dict("index")

merged = pd.merge(s1_subset, gt_subset, left_on="entity_id", right_on="source1_entity_id")
valid_pairs = merged[merged["matched_entity_ids"].notna() & (merged["matched_entity_ids"] != "")]

observed_examples = []
for _, row in valid_pairs.iterrows():
    matches = [m.strip() for m in row["matched_entity_ids"].split(",")]
    for mid in matches:
        target_dict = s2_dict if mid.startswith("S2-") else s3_dict
        if mid in target_dict:
            m = target_dict[mid]
            observed_examples.append({
                "S1_ID": row["entity_id"],
                "Match_ID": mid,
                "Country": row["country"],
                "S1_Name": row["business_name"],
                "Match_Name": m.get("business_name"),
                "S1_Address": row["business_address"],
                "Match_Address": m.get("business_address"),
            })
    if len(observed_examples) >= 8:
        break

obs_df = pd.DataFrame(observed_examples)
obs_df[["Country", "S1_Name", "Match_Name", "S1_Address", "Match_Address"]]


## 8. Demonstrating Preprocessing Functions (`src/preprocessing.py`)

Now we test the modular preprocessing functions implemented in `src/preprocessing.py` on the noisy pairs observed above.


In [ ]:
demo_records = [
    # Legal suffix variation & ampersand
    {"name": "Ramirez, Jackson & Mason Incorporated", "address": "1391 51st Saint, Cleveland, OH", "country": "US"},
    # Domain style name & decorative noise
    {"name": "siiainvestments.com", "address": "##8411 GABRIELINO COURT, PMB 9239, CA", "country": "united states"},
    # Indian legal forms & state abbreviations
    {"name": "FINE TECHNOLOGIES PRIVATE LIMITED", "address": "New No 5 (Old No 2-A) 6th St, Chennai, TN", "country": "ind"},
    # French entity with accents & decorative brackets
    {"name": "<< Team Ecole >> SARL", "address": "175 Boulevard du Pr\ufffdsident Franklin Roosevelt, Bordeaux", "country": "fr"},
    # Missing address edge case
    {"name": "Cardiology Care Enterprises", "address": np.nan, "country": "US"}
]

demo_df = pd.DataFrame(demo_records)
cleaned_demo_df = preprocess_dataframe(demo_df)

display(cleaned_demo_df[[
    "business_name", "clean_business_name",
    "business_address", "clean_business_address",
    "country", "clean_country"
]])


## 9. Summary of Key Findings for Subsequent Phases

### For Phase 2 (Candidate Generation & Blocking):
1. **Address Missingness Handling:**
   - Over 3.3% of Source 2 and Source 3 addresses are `NaN`.
   - **Blocking rule:** Never block exclusively on address tokens (e.g. zipcode, city) alone, or entities with missing addresses will never be paired. Always include name-based blocking keys (e.g. first token, sorted token prefixes, 3-grams) in disjunction (`OR`).
2. **Country Partitioning:**
   - Country partition is an exact blocking attribute (US entities only match US entities; India matches India; France matches France). Blocking strictly within `country` reduces the candidate search space by ~60%.
3. **Word Order Invariance:**
   - Address components are frequently permuted (e.g. `"Boston, 31 Floyd St"` vs `"31 Floyd St, Boston"`). Blocking on unordered token sets or sorted tokens is essential.

### For Phase 3 (Feature Engineering):
1. **Multi-Faceted Similarity Metrics:**
   - Compute token-level overlap (Jaccard, Dice, Overlap coefficient).
   - Compute edit-distance metrics (Levenshtein, Jaro-Winkler) for handling spelling typos.
   - Compute character n-gram TF-IDF cosine similarity for handling partial name matches.
2. **Sub-Field Comparison:**
   - Extract numerical tokens (street numbers, PIN codes) separately. If street numbers disagree, penalize the match.
   - Separate legal suffixes from the core entity name to avoid false positives between distinct businesses with the same legal structure.

---
**Phase 1 Data Understanding & Preprocessing Complete.**
